# 05 - Klasifikasi Konten Kesehatan

Tahap validasi konten kesehatan (3.1.2.4) pada proposal TA. Menentukan video mana yang benar-benar
membahas topik kesehatan (bukan semua konten influencer kesehatan selalu tentang kesehatan).

**Pendekatan 2 fase:**

**FASE A -- Bangun & evaluasi model klasifikasi (SEKALI JALAN, sudah selesai):**
1. `prepare_labeling_sample()` -> ambil sample video buat dilabel manual.
2. **[MANUAL, di luar notebook ini]** -- label video via web app interaktif.
3. `train_classifier_patterns()` -> pelajari pola per-influencer dari 70% data berlabel (train),
   lalu uji ke 30% sisanya (test) pakai Ollama (qwen2.5:7b) + prompt yang disesuaikan pola tiap
   influencer.

**FASE B -- Klasifikasi produksi (otomatis untuk semua video):**
4. `classify_all_videos()` -> terapkan `learned_patterns.json` ke SELURUH
   `metadata_video_konten.csv`. Video dgn confidence rendah/caption kosong/ambigu ditandai
   `check_this=True` untuk direview manual.
5. **[MANUAL, di luar notebook ini]** -- review video `check_this=True`.
6. `export_health_videos()` -> pisahkan `health_videos.csv` vs `not_health_videos.csv`.

**Konsolidasi dari:**
- `01_1_ambil_sample.py` + `01_2_tambah_sample.py` -> `prepare_labeling_sample()`
- `03_test_model_labeling.py` -> `train_classifier_patterns()`
- `05_klasifikasi_videoid.py` -> `classify_all_videos()`
- `07_summary.py` -> `export_health_videos()`

**Sengaja tidak diikutkan (dicek dulu apakah necessary):**
- `02_1_label_manual_sample.py` & `02_2_label_manual_hp.py` (web app Flask interaktif) -- labelnya
  (1999 video) SUDAH FINAL, dicopy ke `output/05_klasifikasi_konten_kesehatan/`.
- `06_klasifikasi_manual.py` (web app review manual `check_this=True`) -- SEMUA 462 video
  `check_this=True` SUDAH direview (0 pending). Tidak perlu dijalankan.
- `03_0_spli_data.py` -- DIHAPUS, 100% redundant dengan `_validate_split()` di bawah.
- Analisis akurasi & visualisasi (`03_01_evaluasi.py`, `04_cek_akurasi.py`) -- jadi bagian
  "Stats/Grafik" di bawah, ditampilkan langsung, tidak menulis file report/plot baru.

**Catatan eksekusi:** `train_classifier_patterns()` & `classify_all_videos()` manggil Ollama per
video (lambat, bisa puluhan menit - berjam-jam). Skip otomatis kalau output sudah ada.

In [ ]:
import json
import re
import time
from collections import defaultdict
from pathlib import Path

import ollama
import pandas as pd

# -------------------------------------------------------------------------
# KONFIGURASI PATH
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "05_klasifikasi_konten_kesehatan"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_METADATA = FINAL_OUTPUT_ROOT / "04_scraping_metadata_konten" / "metadata_video_konten.csv"

VIDEOS_TO_LABEL = OUTPUT_DIR / "videos_to_label.csv"
LABELED_VIDEOS = OUTPUT_DIR / "labeled_videos.csv"          # ground truth manual (IRREPLACEABLE)
TRAIN_SET = OUTPUT_DIR / "train_set.csv"
TEST_SET = OUTPUT_DIR / "test_set.csv"
LEARNED_PATTERNS = OUTPUT_DIR / "learned_patterns.json"
ACCURACY_REPORT = OUTPUT_DIR / "accuracy_report.csv"         # prediksi test set, dipakai cell evaluasi
LABELED_METADATA_VIDEO = OUTPUT_DIR / "labeled_metadata_video.csv"  # hasil klasifikasi SEMUA video
HEALTH_VIDEOS = OUTPUT_DIR / "health_videos.csv"
NOT_HEALTH_VIDEOS = OUTPUT_DIR / "not_health_videos.csv"

TRAIN_SPLIT_RATIO = 0.7
OLLAMA_MODEL = "qwen2.5:7b"

# Throttle CPU (sama utk training & klasifikasi produksi)
OLLAMA_NUM_THREAD = 8
DELAY_BETWEEN_REQUESTS = 3.0
COOLDOWN_EVERY_N_VIDEOS = 500
COOLDOWN_DURATION = 30

SAMPLE_SIZE_PER_BATCH = 1000

MEDICAL_KEYWORDS = ['dokter', 'kesehatan', 'obat', 'penyakit', 'gejala', 'medis',
                     'kesehatan mental', 'diet', 'nutrisi', 'olahraga', 'fitness',
                     'psikolog', 'bidan', 'perawat', 'operasi', 'terapi', 'resep']
NON_HEALTH_KEYWORDS = ['fashion', 'makeup', 'skincare', 'estetika', 'cantik', 'glow',
                        'ootd', 'styling', 'trending', 'viral', 'challenge', 'haul']
AMBIGUOUS_KEYWORDS = ['kulit', 'kecantikan', 'diet', 'sehat', 'kurus', 'gemuk', 'berat badan']

PROMPT_BASE = """Kamu adalah sistem klasifikasi akademik untuk penelitian tugas akhir tentang kategorisasi konten edukasi kesehatan di TikTok Indonesia. Tugasmu murni melabeli data, bukan mendiskusikan isinya.

DEFINISI "KONTEN KESEHATAN":
- Kondisi medis, penyakit, gejala, pengobatan (fisik & mental)
- Kesehatan reproduksi & seksual (menstruasi, kehamilan, KB, kesehatan pria/wanita, edukasi seksual)
- Kesehatan mental (stress, trauma, parenting, kesehatan mental anak-remaja-dewasa)
- Nutrisi, diet, pola makan untuk tujuan kesehatan
- Olahraga/fitness dengan tujuan kesehatan
- Edukasi dari tenaga medis (dokter segala spesialis, psikolog, bidan, perawat)
- Perawatan tubuh/kulit dengan tujuan medis

BUKAN kesehatan:
- Fashion/makeup murni estetika
- Vlog harian tanpa unsur medis
- Hiburan murni, gaming, teknologi, finansial, travel non-kesehatan"""
print("Config siap.")

## FASE A, STEP 1 - Sampling untuk labeling manual

In [ ]:
def prepare_labeling_sample(n_total: int = 2000) -> None:
    """Ambil sample proporsional per influencer untuk dilabel manual.

    Gabungan 01_1_ambil_sample.py (sample pertama, proporsional) dan
    01_2_tambah_sample.py (nambah sample dari influencer yang belum
    ke-cover). Skip kalau videos_to_label.csv sudah ada."""
    if VIDEOS_TO_LABEL.exists():
        print(f"[Sampling] Sudah ada -> {VIDEOS_TO_LABEL} (skip, pakai sample lama)")
        return

    if not INPUT_METADATA.exists():
        raise FileNotFoundError(f"File input tidak ditemukan: {INPUT_METADATA}")

    df = pd.read_csv(INPUT_METADATA, dtype={"video_id": str})
    unique_influencers = df["username"].unique()

    sample_dfs = []
    for influencer in unique_influencers:
        inf_videos = df[df["username"] == influencer]
        target_samples = max(1, int((len(inf_videos) / len(df)) * n_total))
        actual_samples = min(target_samples, len(inf_videos))
        sample_dfs.append(inf_videos.sample(n=actual_samples, random_state=42))

    sampled_df = pd.concat(sample_dfs, ignore_index=True)

    if len(sampled_df) < n_total:
        shortage = n_total - len(sampled_df)
        sampled_ids = set(sampled_df["video_id"].astype(str))
        remaining = df[~df["video_id"].astype(str).isin(sampled_ids)]
        if len(remaining) >= shortage:
            sampled_df = pd.concat(
                [sampled_df, remaining.sample(n=shortage, random_state=42)], ignore_index=True
            )

    sampled_df = sampled_df.sample(frac=1, random_state=42).reset_index(drop=True)
    sampled_df["manual_label"] = ""
    sampled_df["label_timestamp"] = ""
    sampled_df.to_csv(VIDEOS_TO_LABEL, index=False)

    print(f"[Sampling] {len(sampled_df)} video disiapkan untuk label manual -> {VIDEOS_TO_LABEL}")
    print("[Sampling] Lanjutkan dengan label manual (lihat markdown di atas).")


prepare_labeling_sample()

## [MANUAL] Label video via web app interaktif
Hasil label manual (1999 video) SUDAH FINAL -> `labeled_videos.csv`. Lanjut ke cell berikutnya.

## Helper bersama (dipakai training & klasifikasi produksi)

In [ ]:
def _analyze_caption(caption) -> dict:
    caption_str = str(caption).lower() if pd.notna(caption) else ""
    return {
        "length": len(caption_str),
        "has_medical_terms": any(term in caption_str for term in MEDICAL_KEYWORDS),
        "medical_term_count": sum(caption_str.count(term) for term in MEDICAL_KEYWORDS),
        "has_non_health_terms": any(term in caption_str for term in NON_HEALTH_KEYWORDS),
        "ambiguous_term_count": sum(caption_str.count(term) for term in AMBIGUOUS_KEYWORDS),
    }


def _build_prompt(description: str, username: str, health_pct: float) -> str:
    prompt = PROMPT_BASE
    if health_pct > 70:
        prompt += f"\n\nKONTEKS: @{username} biasanya upload konten kesehatan ({health_pct:.0f}%)."
    elif health_pct < 30:
        prompt += f"\n\nKONTEKS: @{username} jarang upload konten kesehatan ({health_pct:.0f}%). Hati-hati jangan false positive."
    else:
        prompt += f"\n\nKONTEKS: @{username} upload balanced health & non-health ({health_pct:.0f}%)."
    prompt += f"\n\nCaption video:\n\"{description}\"\n\nJawab HANYA dalam format ini:\nLABEL: HEALTH atau NOT_HEALTH\nCONFIDENCE: angka 0.0-1.0"
    return prompt


def _parse_ollama_response(response_text: str):
    try:
        label_match = re.search(r'LABEL:\s*(HEALTH|NOT_HEALTH)', response_text, re.IGNORECASE)
        if not label_match:
            return False, 0.5
        is_health = label_match.group(1).upper() == "HEALTH"
        conf_match = re.search(r'CONFIDENCE:\s*([\d.]+)', response_text)
        confidence = float(conf_match.group(1)) if conf_match else 0.5
        return is_health, max(0.0, min(1.0, confidence))
    except Exception:
        return False, 0.5


def _classify_one(description: str, username: str, health_pct: float, ollama_call_count: int):
    """Classify 1 video. Return (is_health, confidence, method, ollama_call_count_baru)."""
    if len(description) < 5:
        is_health = health_pct > 50
        confidence = abs(health_pct - 50) / 100
        return is_health, confidence, "no_caption_learned", ollama_call_count

    try:
        prompt = _build_prompt(description, username, health_pct)
        response = ollama.generate(
            model=OLLAMA_MODEL, prompt=prompt, stream=False,
            options={"temperature": 0.1, "num_predict": 100, "num_thread": OLLAMA_NUM_THREAD},
        )
        is_health, confidence = _parse_ollama_response(response["response"])
        ollama_call_count += 1
        time.sleep(DELAY_BETWEEN_REQUESTS)
        if ollama_call_count % COOLDOWN_EVERY_N_VIDEOS == 0:
            print(f"   Cooldown {COOLDOWN_DURATION}s (sudah {ollama_call_count} panggilan Ollama)...")
            time.sleep(COOLDOWN_DURATION)
        return is_health, confidence, "ollama_enhanced", ollama_call_count
    except Exception as e:
        print(f"   Error Ollama: {e}")
        return False, 0.5, "error", ollama_call_count

## FASE A, STEP 3 - Training: pelajari pola per-influencer + evaluasi di test set

In [ ]:
def _validate_split(train_df, test_df, original_df) -> bool:
    """Pastikan tiap influencer (>=2 video) punya data train DAN test."""
    valid_influencers = {
        u for u in original_df["username"].unique()
        if len(original_df[original_df["username"] == u]) >= 2
    }
    train_inf = set(train_df["username"].unique())
    test_inf = set(test_df["username"].unique())
    missing = (valid_influencers - train_inf) | (valid_influencers - test_inf)
    if missing:
        print(f"[Training] Split tidak valid, influencer tanpa train/test: {missing}")
        return False
    return True


def train_classifier_patterns() -> None:
    """Split 70/30 per influencer, pelajari pola dari train set, uji ke test set + Ollama.

    Skip kalau learned_patterns.json & accuracy_report.csv sudah ada."""
    if LEARNED_PATTERNS.exists() and ACCURACY_REPORT.exists():
        print(f"[Training] Sudah ada -> {LEARNED_PATTERNS.name} & {ACCURACY_REPORT.name} (skip training berat)")
        return

    if not LABELED_VIDEOS.exists():
        raise FileNotFoundError(
            f"File ground truth tidak ditemukan: {LABELED_VIDEOS}. "
            f"Perlu hasil label manual (lihat markdown di atas)."
        )

    try:
        ollama.list()
    except Exception:
        raise RuntimeError("Ollama belum jalan. Jalankan 'ollama serve' dulu.")

    df = pd.read_csv(LABELED_VIDEOS, dtype={"video_id": str})
    df_labeled = df[df["manual_label"].notna() & (df["manual_label"] != "")].copy()
    print(f"[Training] {len(df_labeled)} video berlabel manual, {df_labeled['username'].nunique()} influencer")

    # Split 70/30 stratified per influencer
    train_dfs, test_dfs = [], []
    for username in sorted(df_labeled["username"].unique()):
        inf_data = df_labeled[df_labeled["username"] == username]
        n = len(inf_data)
        if n < 2:
            continue
        split_idx = max(1, int(n * TRAIN_SPLIT_RATIO))
        split_idx = min(split_idx, n - 1)
        shuffled = inf_data.sample(frac=1, random_state=42)
        train_dfs.append(shuffled.iloc[:split_idx])
        test_dfs.append(shuffled.iloc[split_idx:])

    train_df = pd.concat(train_dfs, ignore_index=True)
    test_df = pd.concat(test_dfs, ignore_index=True)

    if not _validate_split(train_df, test_df, df_labeled):
        raise ValueError("Split 70/30 gagal validasi integritas (ada influencer tanpa train/test).")

    train_df.to_csv(TRAIN_SET, index=False)
    test_df.to_csv(TEST_SET, index=False)
    print(f"[Training] Split tersimpan: {len(train_df)} train / {len(test_df)} test")

    # Learn pola per-influencer dari train set
    learnings = {}
    for username in train_df["username"].unique():
        inf_data = train_df[train_df["username"] == username]
        health_count = (inf_data["manual_label"] == "health").sum()
        no_caption = inf_data["description"].fillna("").str.len() < 5
        learnings[username] = {
            "health_pct": health_count / len(inf_data) * 100,
            "no_caption_pct": no_caption.sum() / len(inf_data) * 100,
            "health_samples": [str(s)[:100] for s in inf_data[inf_data["manual_label"] == "health"]["description"].head(3) if pd.notna(s)],
            "non_health_samples": [str(s)[:100] for s in inf_data[inf_data["manual_label"] == "not_health"]["description"].head(3) if pd.notna(s)],
        }
    print(f"[Training] Pola dipelajari untuk {len(learnings)} influencer")

    # Classify test set + track errors
    predictions, errors_global, errors_per_inf = [], {"fn": [], "fp": []}, defaultdict(lambda: {"fn": [], "fp": []})
    ollama_call_count = 0
    total = len(test_df)
    print(f"[Training] Klasifikasi {total} video test set (bisa lama, ada panggilan Ollama)...")

    for idx, (_, row) in enumerate(test_df.iterrows(), 1):
        description = str(row["description"]).strip()[:800] if pd.notna(row["description"]) else ""
        health_pct = learnings.get(row["username"], {}).get("health_pct", 50)
        is_health, confidence, method, ollama_call_count = _classify_one(
            description, row["username"], health_pct, ollama_call_count
        )
        predicted_label = "health" if is_health else "not_health"
        is_correct = predicted_label == row["manual_label"]

        predictions.append({
            "username": row["username"], "video_id": row["video_id"], "description": description[:100],
            "predicted_label": predicted_label, "predicted_confidence": confidence,
            "actual_label": row["manual_label"], "correct": is_correct, "method": method,
        })

        if not is_correct:
            caption_chars = _analyze_caption(description)
            entry = {"caption": description[:150], "confidence": confidence, **caption_chars}
            if row["manual_label"] == "health":
                errors_global["fn"].append(entry)
                errors_per_inf[row["username"]]["fn"].append(entry)
            else:
                errors_global["fp"].append(entry)
                errors_per_inf[row["username"]]["fp"].append(entry)

        if idx % 50 == 0:
            print(f"   Progress: {idx}/{total}")

    predictions_df = pd.DataFrame(predictions)
    predictions_df.to_csv(ACCURACY_REPORT, index=False)

    # Evaluate
    accuracy = predictions_df["correct"].mean() * 100
    pred_health = predictions_df["predicted_label"] == "health"
    actual_health = predictions_df["actual_label"] == "health"
    tp, fp = (pred_health & actual_health).sum(), (pred_health & ~actual_health).sum()
    tn, fn = (~pred_health & ~actual_health).sum(), (~pred_health & actual_health).sum()
    precision = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    per_inf_accuracy = {
        u: round(predictions_df[predictions_df["username"] == u]["correct"].mean() * 100, 1)
        for u in predictions_df["username"].unique()
    }

    # Save learned patterns (dipakai classify_all_videos + cell evaluasi di bawah)
    patterns_data = {
        "metadata": {
            "overall_accuracy": accuracy, "overall_precision": precision,
            "overall_recall": recall, "overall_f1": f1,
        },
        "influencers": {
            u: {
                "health_percentage": round(info["health_pct"], 1),
                "no_caption_percentage": round(info["no_caption_pct"], 1),
                "accuracy": per_inf_accuracy.get(u, 0),
                "health_samples": info["health_samples"],
                "non_health_samples": info["non_health_samples"],
                "false_negatives_count": len(errors_per_inf[u]["fn"]),
                "false_positives_count": len(errors_per_inf[u]["fp"]),
            }
            for u, info in learnings.items()
        },
    }
    with open(LEARNED_PATTERNS, "w", encoding="utf-8") as f:
        json.dump(patterns_data, f, ensure_ascii=False, indent=2)

    print(f"\n[Training] SELESAI. Accuracy={accuracy:.1f}% Precision={precision:.1f}% Recall={recall:.1f}% F1={f1:.1f}%")
    print(f"[Training] Tersimpan -> {LEARNED_PATTERNS.name}, {ACCURACY_REPORT.name}")


train_classifier_patterns()

## FASE B, STEP 4 - Klasifikasi produksi: terapkan ke SEMUA video

In [ ]:
def _is_ambiguous_caption(description: str) -> bool:
    if not description or len(description) < 10:
        return True
    stripped = description.strip()
    if stripped.startswith("#") or stripped.startswith("@"):
        return True
    if ("#stitch" in stripped.lower() or "#duet" in stripped.lower()) and len(stripped) < 30:
        return True
    return False


def classify_all_videos() -> None:
    """Terapkan learned_patterns.json ke SEMUA video di metadata_video_konten.csv.

    Resume-aware (skip video yang sudah ada di output). Video manual_label yang sudah ada
    (dari labeled_videos.csv) langsung dipakai tanpa Ollama. Video confidence rendah/caption
    kosong/ambigu ditandai check_this=True."""
    if not LEARNED_PATTERNS.exists():
        raise FileNotFoundError(f"{LEARNED_PATTERNS} tidak ada. Jalankan train_classifier_patterns() dulu.")

    with open(LEARNED_PATTERNS, "r", encoding="utf-8") as f:
        patterns = json.load(f)

    df_metadata = pd.read_csv(INPUT_METADATA, dtype={"video_id": str})
    df_metadata["manual_label"] = ""

    if LABELED_VIDEOS.exists():
        df_labeled = pd.read_csv(LABELED_VIDEOS, dtype={"video_id": str})
        labeled_map = {
            (r["username"], r["video_id"]): r["manual_label"]
            for _, r in df_labeled.iterrows() if pd.notna(r.get("manual_label")) and r["manual_label"]
        }
        for idx, row in df_metadata.iterrows():
            key = (row["username"], row["video_id"])
            if key in labeled_map:
                df_metadata.at[idx, "manual_label"] = labeled_map[key]
        print(f"[Klasifikasi] {len(labeled_map)} video sudah punya label manual, dipakai langsung")

    existing_ids = set()
    if LABELED_METADATA_VIDEO.exists():
        existing_df = pd.read_csv(LABELED_METADATA_VIDEO, dtype={"username": str, "video_id": str})
        existing_ids = set(zip(existing_df["username"], existing_df["video_id"]))
        print(f"[Klasifikasi] {len(existing_ids)} video sudah ada di output (resume)")

    labeled = df_metadata[df_metadata["manual_label"] != ""].copy()
    unlabeled = df_metadata[df_metadata["manual_label"] == ""].copy()

    def _already_done(row):
        return (row["username"], row["video_id"]) in existing_ids

    labeled_pending = labeled[~labeled.apply(_already_done, axis=1)]
    unlabeled_pending = unlabeled[~unlabeled.apply(_already_done, axis=1)]

    if len(labeled_pending) == 0 and len(unlabeled_pending) == 0:
        print(f"[Klasifikasi] Semua {len(df_metadata)} video sudah diklasifikasi, skip.")
        return

    if not LABELED_METADATA_VIDEO.exists():
        pd.DataFrame(columns=["username", "video_id", "description", "predicted_label",
                               "predicted_confidence", "check_this", "method"]).to_csv(
            LABELED_METADATA_VIDEO, index=False, encoding="utf-8"
        )

    # Tulis dulu video yang sudah punya label manual (gratis, tanpa Ollama)
    for _, row in labeled_pending.iterrows():
        row_dict = {
            "username": row["username"], "video_id": row["video_id"],
            "description": str(row["description"])[:100] if pd.notna(row["description"]) else "",
            "predicted_label": row["manual_label"], "predicted_confidence": 1.0,
            "check_this": False, "method": "manual_labeled",
        }
        pd.DataFrame([row_dict]).to_csv(LABELED_METADATA_VIDEO, mode="a", header=False, index=False, encoding="utf-8")

    print(f"[Klasifikasi] {len(unlabeled_pending)} video perlu diklasifikasi otomatis...")
    ollama_call_count = n_classified = 0

    for idx, (_, row) in enumerate(unlabeled_pending.iterrows(), 1):
        description = str(row["description"]).strip()[:800] if pd.notna(row["description"]) else ""
        has_caption = len(description) >= 5
        inf_data = patterns["influencers"].get(row["username"], {})
        influencer_accuracy = inf_data.get("accuracy", 50)
        health_pct = inf_data.get("health_percentage", 50)

        is_health, confidence, method, ollama_call_count = _classify_one(
            description, row["username"], health_pct, ollama_call_count
        )

        n_false = sum([influencer_accuracy < 80, confidence < 0.70, not has_caption])
        check_this = (n_false > 1) or _is_ambiguous_caption(description)

        row_dict = {
            "username": row["username"], "video_id": row["video_id"], "description": description[:100],
            "predicted_label": "health" if is_health else "not_health",
            "predicted_confidence": round(confidence, 3), "check_this": check_this, "method": method,
        }
        pd.DataFrame([row_dict]).to_csv(LABELED_METADATA_VIDEO, mode="a", header=False, index=False, encoding="utf-8")
        n_classified += 1

        if idx % 50 == 0:
            print(f"   Progress: {idx}/{len(unlabeled_pending)} | Tersimpan: {n_classified}")

    result_df = pd.read_csv(LABELED_METADATA_VIDEO)
    n_check = (result_df["check_this"] == True).sum()
    print(f"\n[Klasifikasi] SELESAI. Total: {len(result_df)} | perlu review manual (check_this=True): {n_check}")
    if n_check > 0:
        print("[Klasifikasi] Ada video check_this=True yang belum direview -- "
              "jalankan 01_pengumpulan_data/04_2_Klasifikasikan_konten_scraping/06_klasifikasi_manual.py "
              "(web app interaktif) untuk review manual.")


classify_all_videos()

## [MANUAL] Review video check_this=True
SEMUA 462 video `check_this=True` SUDAH direview. Kalau ada video baru dengan `check_this=True`,
jalankan `06_klasifikasi_manual.py` (web app) sebelum lanjut ke cell berikutnya.

## FASE B, STEP 6 - Export final: health_videos.csv / not_health_videos.csv

In [ ]:
def export_health_videos() -> None:
    if HEALTH_VIDEOS.exists() and NOT_HEALTH_VIDEOS.exists():
        print(f"[Export] Sudah ada -> {HEALTH_VIDEOS.name} & {NOT_HEALTH_VIDEOS.name} (skip)")
        return

    if not LABELED_METADATA_VIDEO.exists():
        raise FileNotFoundError(f"{LABELED_METADATA_VIDEO} tidak ada. Jalankan classify_all_videos() dulu.")

    df = pd.read_csv(LABELED_METADATA_VIDEO)
    health_df = df[df["predicted_label"] == "health"].copy()
    not_health_df = df[df["predicted_label"] == "not_health"].copy()

    health_df.to_csv(HEALTH_VIDEOS, index=False, encoding="utf-8")
    not_health_df.to_csv(NOT_HEALTH_VIDEOS, index=False, encoding="utf-8")

    print(f"[Export] {len(health_df)} health videos -> {HEALTH_VIDEOS.name}")
    print(f"[Export] {len(not_health_df)} not-health videos -> {NOT_HEALTH_VIDEOS.name}")
    print(f"[Export] Rasio health: {len(health_df) / len(df) * 100:.1f}%")


export_health_videos()

## Stats/Grafik -- Evaluasi Model Klasifikasi Konten Kesehatan

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

assert LEARNED_PATTERNS.exists(), f"{LEARNED_PATTERNS} belum ada -- jalankan train_classifier_patterns() dulu"
assert ACCURACY_REPORT.exists(), f"{ACCURACY_REPORT} belum ada -- jalankan train_classifier_patterns() dulu"

with open(LEARNED_PATTERNS, "r", encoding="utf-8") as f:
    patterns = json.load(f)

predictions_df = pd.read_csv(ACCURACY_REPORT)
print(f"Loaded {len(predictions_df)} prediksi test set, {len(patterns['influencers'])} influencer")

### 1. Metrik Keseluruhan (Test Set)

In [ ]:
pred_health = predictions_df["predicted_label"] == "health"
actual_health = predictions_df["actual_label"] == "health"

tp = int((pred_health & actual_health).sum())
fp = int((pred_health & ~actual_health).sum())
tn = int((~pred_health & ~actual_health).sum())
fn = int((~pred_health & actual_health).sum())

total = len(predictions_df)
accuracy = predictions_df["correct"].mean() * 100
precision = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
specificity = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0

print(f"Total video diuji : {total}")
print(f"Accuracy           : {accuracy:.1f}%")
print(f"Precision          : {precision:.1f}%")
print(f"Recall             : {recall:.1f}%")
print(f"F1-Score           : {f1:.1f}%")
print(f"Specificity        : {specificity:.1f}%")

### 2. Confusion Matrix & Grafik Metrik

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Model Performance Metrics (Test Set)", fontsize=16, fontweight="bold")

ax = axes[0, 0]
ax.bar(["Accuracy"], [accuracy], color="#667eea", width=0.6)
ax.set_ylim(0, 100); ax.set_ylabel("Percentage (%)"); ax.set_title("Overall Accuracy")
ax.text(0, accuracy + 2, f"{accuracy:.1f}%", ha="center", fontweight="bold")

ax = axes[0, 1]
vals = [precision, recall, f1]
bars = ax.bar(["Precision", "Recall", "F1-Score"], vals, color=["#51cf66", "#ff922b", "#a78bfa"], width=0.6)
ax.set_ylim(0, 100); ax.set_ylabel("Percentage (%)"); ax.set_title("Precision, Recall, F1-Score")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=9)

ax = axes[1, 0]
bars = ax.bar(["Recall (Sensitivity)", "Specificity"], [recall, specificity], color=["#51cf66", "#ff6b6b"], width=0.6)
ax.set_ylim(0, 100); ax.set_ylabel("Percentage (%)"); ax.set_title("Sensitivity vs Specificity")
for b, v in zip(bars, [recall, specificity]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=9)

ax = axes[1, 1]
sns.heatmap([[tp, fp], [fn, tn]], annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Predicted Health", "Predicted Non-Health"],
            yticklabels=["Actual Health", "Actual Non-Health"])
ax.set_title("Confusion Matrix (Absolute Numbers)")

plt.tight_layout()
plt.show()

### 3. Ranking Akurasi per Influencer

In [ ]:
per_inf = []
for username in predictions_df["username"].unique():
    inf_df = predictions_df[predictions_df["username"] == username]
    per_inf.append({
        "username": username,
        "test_videos": len(inf_df),
        "accuracy": round(inf_df["correct"].mean() * 100, 1),
        "correct": int(inf_df["correct"].sum()),
    })

per_inf_df = pd.DataFrame(per_inf).sort_values("accuracy", ascending=False).reset_index(drop=True)

print("Top 10 akurasi tertinggi:")
print(per_inf_df.head(10))

print("\nBottom 10 akurasi terendah:")
print(per_inf_df.tail(10).sort_values("accuracy"))

### 4. Analisis Pola Error (False Negative / False Positive)

In [ ]:
fn_df = predictions_df[(predictions_df["actual_label"] == "health") & (predictions_df["predicted_label"] == "not_health")]
fp_df = predictions_df[(predictions_df["actual_label"] == "not_health") & (predictions_df["predicted_label"] == "health")]

print(f"False Negative (health terlewat): {len(fn_df)}")
print(f"False Positive (non-health salah diprediksi health): {len(fp_df)}")

if len(fn_df) > len(fp_df) * 2:
    print("\n-> Model cenderung TERLALU KONSERVATIF (banyak melewatkan konten health).")
elif len(fp_df) > len(fn_df) * 2:
    print("\n-> Model cenderung TERLALU AGRESIF (banyak salah menandai non-health sebagai health).")
else:
    print("\n-> Error FN dan FP relatif seimbang.")

print("\nInfluencer dengan isu FN terbanyak:")
print(fn_df["username"].value_counts().head(5))

print("\nInfluencer dengan isu FP terbanyak:")
print(fp_df["username"].value_counts().head(5))

### 5. Breakdown Hasil Klasifikasi Final (Seluruh Dataset)

In [ ]:
if HEALTH_VIDEOS.exists() and NOT_HEALTH_VIDEOS.exists():
    health_df = pd.read_csv(HEALTH_VIDEOS)
    not_health_df = pd.read_csv(NOT_HEALTH_VIDEOS)
    total_all = len(health_df) + len(not_health_df)

    print(f"Total video diklasifikasi : {total_all}")
    print(f"Health                    : {len(health_df)} ({len(health_df)/total_all*100:.1f}%)")
    print(f"Not-health                : {len(not_health_df)} ({len(not_health_df)/total_all*100:.1f}%)")

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie([len(health_df), len(not_health_df)], labels=["Health", "Not-health"],
           autopct="%1.1f%%", colors=["#51cf66", "#ff922b"], startangle=90)
    ax.set_title("Proporsi Konten Kesehatan vs Non-Kesehatan")
    plt.show()

    print("\nContoh video health (3 teratas):")
    print(health_df[["username", "description", "predicted_confidence", "check_this"]].head(3))
else:
    print("health_videos.csv / not_health_videos.csv belum ada -- jalankan export_health_videos() dulu")